In [1]:
import pandas as pd
import numpy as np
import os
import json
import difflib

### get the dictionary dimensions and values to verify schema

In [2]:
path='C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR'
df_translation=pd.read_excel(path+'/translation dict.xlsx')

''' it will create such a structure
{
    "Country": {
        "dim_values": {
            "Egypt": "مصر",
            "Lebanon": "لبنان"
        },
        "dim": {
            "Country": "البلد"
        }
    }
}'''

En_Ar_dictionary={}
#get the unique dimensions
dimensions = set(df_translation['col_en'].unique())

for dim in dimensions:
    df_dim=df_translation[df_translation['col_en'].isin([dim.lower(), dim])].copy()
    En_Ar_dictionary.update(
        {dim:{'dim_values':dict(zip(df_dim['val_en'], df_dim['val_ar'])), 
                'dim': {df_dim['col_en'].unique()[0]:df_dim['col_ar'].unique()[0]}}})

Ar_En_dictionary={}
#get the unique dimensions
dimensions = set(df_translation['col_ar'].unique())

for dim in dimensions:
    df_dim=df_translation[df_translation['col_ar'].isin([dim.lower(), dim])].copy()
    Ar_En_dictionary.update(
        {dim:{'dim_values':dict(zip(df_dim['val_ar'], df_dim['val_en'])), 
                'dim': {df_dim['col_ar'].unique()[0]:df_dim['col_en'].unique()[0]}}})

In [3]:
# folder_path='C:/Users/RSHIRINI/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR/datacollector_received_quest/recieved_quests'

folder_path='C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR/datacollector_received_quest'
# folder_path='C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR'
#check all xlsx files in the folder

# Get all excel files in the directory
xlsx_files = [f for f in os.listdir(folder_path) if f.endswith('.xlsx')]

xlsx_files

['Copy of Copy of exports_Libya health_amended.xlsx',
 'exports_Iraq housing_20260507_144743.xlsx',
 'exports_Iraq poverty_20260507_144645.xlsx',
 'exports_Jordan housing_20260525_051245.xlsx',
 'exports_Labor Iraq_20260507_144525.xlsx',
 'exports_Labor Libya_20260514_114204.xlsx',
 'exports_Libya poverty_20260511_093943.xlsx',
 'exports_Tunisia education_20260515_113641.xlsx',
 'exports_Tunisia health_20260521_070817.xlsx',
 'exports_Tunisia housing_20260518_052511.xlsx',
 'exports_Tunisia labor new_20260515_113507.xlsx',
 'exports_Tunisia poverty_20260505_074030.xlsx']

In [ ]:
dataframes = []

# Added 'الفصل' to keep it from being changed/fixed
standard_cols = {'السنة', 'العدد', 'المصدر', 'الفصل'}

category_map = {
    'housing': 'سكن',
    'population': 'سكان',
    'labor': 'عمالة',
    'education': 'التعليم',
    'poverty': 'الفقر',
    'health': 'الصحة'
}

for file in xlsx_files:
    print(f'--- Opening Workbook: {file} ---')
    xls = pd.ExcelFile(os.path.join(folder_path, file))
    
    for sheet_name in xls.sheet_names:
        df_raw = pd.read_excel(xls, sheet_name=sheet_name, header=None, dtype=str)
        header_rows = df_raw.index[df_raw[0] == 'index'].tolist()
        if len(header_rows) < 2: continue

        # 1. prepare the data and source slices
        df_data = df_raw[df_raw[0] == '1'].copy()
        df_data.columns = df_raw.iloc[header_rows[0]].dropna().str.strip()
        # Drop index now
        df_data = df_data.drop(columns=['index'], errors='ignore')
        
        df_source = df_raw[df_raw[0] == '2'].copy()
        df_source = df_source[df_raw.iloc[header_rows[1]].dropna().str.strip().index]
        df_source.columns = df_raw.iloc[header_rows[1]].dropna().str.strip()
        # Drop index now
        df_source = df_source.drop(columns=['index'], errors='ignore')

        # Extract category and add 'الفصل' column
        sheet_clean = sheet_name.lower().strip()
        raw_cat = sheet_clean.split('_')[0].strip()
            
        # Map "education" straight to "التعليم"
        df_data['الفصل'] = category_map.get(raw_cat, raw_cat)
        
        # 2. IDENTIFY & REPLACE
        search_targets = list(standard_cols) + list(Ar_En_dictionary.keys())
        
        for df_temp in [df_data, df_source]:
            # Fix Columns
            for col in list(df_temp.columns):
                if col in search_targets: continue
                
                match = difflib.get_close_matches(str(col), search_targets, n=1, cutoff=0.6)
                if match:
                    best_match = match[0]
                    # RE-ADDED CHARACTER COUNTS HERE
                    print(f"  >> FIX COL: {col} ({len(str(col))} chars) -> {best_match} ({len(str(best_match))} chars)")
                    df_temp.rename(columns={col: best_match}, inplace=True)

                if df_temp.columns.duplicated().any():
                    print(f"!!! DUPLICATE ERROR on '{best_match}'. STOPPING.")
                    raise SystemExit

            # Fix Values
            for col in df_temp.columns:
                if col in Ar_En_dictionary:
                    # Logic to keep difflib from crashing on non-string data
                    allowed_str = [str(a) for a in Ar_En_dictionary[col]['dim_values'].keys() if pd.notna(a)]
                    
                    for val in df_temp[col].unique():
                        if pd.isna(val) or str(val) in allowed_str: continue
                        
                        v_match = difflib.get_close_matches(str(val), allowed_str, n=1, cutoff=0.6)
                        if v_match:
                            best_v = v_match[0]
                            # RE-ADDED CHARACTER COUNTS HERE
                            print(f"  FIX VAL: [{col}] {val} ({len(str(val))} chars) -> {best_v} ({len(str(best_v))} chars)")
                            df_temp[col] = df_temp[col].replace(val, best_v)


        # --- 3. FINAL MERGE (Clean & Linear) ---
        # id_vars are all non-digit columns (Indicator, Country, Chapter, etc.)
        id_vars = [c for c in df_data.columns if not str(c).isdigit()]
        # year_columns are only the digit columns (2010, 2011, etc.)
        year_columns = [c for c in df_data.columns if str(c).isdigit()]
        
        df_melted = df_data.melt(
            id_vars=id_vars, 
            value_vars=year_columns, 
            var_name='السنة', 
            value_name='العدد'
        )

        # CRASH LOCATOR BLOCK
        try:
            # Merge melted data with source info
            df_merged = pd.merge(df_melted, df_source, on=['السنة', 'المؤشر', 'الدولة'], how='left')
            dataframes.append(df_merged)
            
        except KeyError as e:
            print("\n" + "="*50)
            print(f"💥 CRASH DETECTED IN WORKBOOK: {file}")
            print(f"💥 ON SHEET: {sheet_name}")
            print(f"💥 FOR CHAPTER ('الفصل'): {df_data['الفصل'].iloc[0] if 'الفصل' in df_data.columns else 'Unknown'}")
            print("="*50)
            print(f"Columns actually found in df_melted:\n{list(df_melted.columns)}\n")
            print(f"Columns actually found in df_source:\n{list(df_source.columns)}")
            print("="*50)
            raise e

--- Opening Workbook: Copy of Copy of exports_Libya health_amended.xlsx ---
  FIX VAL: [المؤشر] الولادات التي يشرف عليها أخصائيون صحّيون مَهَرة (نسبة مئوية)) حسب المواطنة (74 chars) -> الولادات التي يشرف عليها أخصائيون صحّيون مَهَرة (نسبة مئوية) حسب المواطنة (73 chars)
  FIX VAL: [الفئة العمرية] 15-24 (5 chars) -> 15-24 سنة (9 chars)
  FIX VAL: [الفئة العمرية] 15-24 (5 chars) -> 15-24 سنة (9 chars)
  FIX VAL: [الفئة العمرية] 15-24 (5 chars) -> 15-24 سنة (9 chars)
  FIX VAL: [الفئة العمرية] 15-24 (5 chars) -> 15-24 سنة (9 chars)
  FIX VAL: [المؤشر] الولادات التي يشرف عليها أخصائيون صحّيون مَهَرة (نسبة مئوية)) حسب المواطنة (74 chars) -> الولادات التي يشرف عليها أخصائيون صحّيون مَهَرة (نسبة مئوية) حسب المواطنة (73 chars)
  FIX VAL: [الفئة العمرية] 15-24 (5 chars) -> 15-24 سنة (9 chars)
  FIX VAL: [الفئة العمرية] 15-24 (5 chars) -> 15-24 سنة (9 chars)
  FIX VAL: [الفئة العمرية] 15-24 (5 chars) -> 15-24 سنة (9 chars)
  FIX VAL: [الفئة العمرية] 15-24 (5 chars) -> 15-24 سنة (9 chars)
--- Open

In [15]:
# Final aggregation
if dataframes:
    final_df = pd.concat(dataframes, ignore_index=True)
    
    # One last safety strip in case different sheets had different trailing spaces 
    # that survived the merge or concat
    final_df.columns = final_df.columns.str.strip()
    
    final_df = final_df.dropna(axis=1, how='all')
    final_df.to_excel('final_dataset_combined.xlsx', index=False)
    print("\nSuccess: Combined data from all workbooks and sheets.")
else:
    print("\nNo data found. Check your folder path and sheet structures.")


Success: Combined data from all workbooks and sheets.


In [16]:
final_df.columns

Index(['المؤشر', 'الدولة', 'المواطنة', 'الفصل', 'السنة', 'العدد', 'المصدر',
       'المنطقة', 'الجنس', 'الفئة العمرية', 'مصدر الإضاءة', 'نوع مكان الإقامة',
       'نوع حيازة الوحدات السكنية', 'مصدر مياه الشرب',
       'أنواع نظام التخلص من مياه الصرف الصحي', 'الفئة',
       'نوع الخدمات/المنتجات', 'وضع العمالة',
       'أسباب البقاء خارج القوى العاملة', 'القطاع المؤسسي',
       'أقسام المهن الرئيسية', 'أقسام النشاط الإقتصادي', 'المرحلة التعليمية'],
      dtype='object')

### check all the column names if they are  unique

In [7]:
import difflib

cols = list(final_df.columns)
threshold = 0.9

for col in cols:
    # Find everything in the list that is "close enough" to the current column
    # By setting it to len(cols), you are telling Python: "Don't stop at 3 (default),
    # show me every single match you find in the entire list."
    matches = difflib.get_close_matches(col, cols, n=len(cols), cutoff=threshold)
    
    # If it found more than itself, you have a duplicate issue
    if len(matches) > 1:
        print(f"Similarity found for '{col}': {matches}")

### check unique values in columns

In [8]:
import difflib

# Keep THRESHOLD very high to only catch typos/spaces, not different indicators
THRESHOLD = 0.98 
fuzzy_report = {}

# Compare each value with every other value in the list
'''enumerate(values): This gives you two things at once: the index i (the position) and the val1 (the actual text, like "الجزائر").
This loop picks the "Anchor" item. We are going to hold this item in our hand and compare it to others.'''
 
for column, values in Ar_En_dictionary.items():
    similar_pairs = []
    
    # FIX: Convert 'values' to a list so Python knows how to slice it properly
    values_list = list(values)

    for i, val1 in enumerate(values_list):
        for val2 in values_list[i+1:]:
            # Use raw strings to catch hidden spaces
            s1, s2 = str(val1), str(val2)
            score = difflib.SequenceMatcher(None, s1, s2).ratio()
            
            #Only show if they are ALMOST identical but NOT exactly 1.0
            if score >= THRESHOLD and score < 1.0:
                # Get the sources
                src1 = final_df[final_df[column] == val1]['file_source'].unique().tolist()
                src2 = final_df[final_df[column] == val2]['file_source'].unique().tolist()
                
                # Format exactly as you requested
                # Wrapping values in quotes "" helps you see the trailing spaces in the terminal
                report_entry = f"\"{val1}\" {src1} <<<--->>> \"{val2}\" {src2}"
                similar_pairs.append(report_entry)
                
    if similar_pairs:
        fuzzy_report[column] = similar_pairs

# Print the diagnostic report
print(json.dumps(fuzzy_report, indent=4, ensure_ascii=False))


{}
